# GM-GReFEL Live Demo (Identical to localhost:5050)
**Authors:** Yudhistira Arditya Pratama, Yi-Zeng Hsieh - NTUST

### Instructions:
1. Runtime -> Change runtime type -> GPU (T4)
2. Run **Cell 1** only -> wait -> **Runtime -> Restart session**
3. Run **Cells 2, 3, 4** -> click the ngrok URL that appears at the bottom

> Login credentials: **username: admin  /  password: 1234**

In [ ]:
# Cell 1 - Install packages
# After this cell finishes -> Runtime -> Restart session -> then run Cells 2,3,4
import subprocess, sys, torch
print(f'Python {sys.version_info.major}.{sys.version_info.minor} | PyTorch {torch.__version__} | CUDA {torch.version.cuda}')

pkgs = ['flask','flask-cors','mediapipe','scipy',
        'opencv-python-headless','gdown','pyngrok',
        'timm','einops','packaging','ninja']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=True)
print('Standard packages: done')

for pkg in ['causal-conv1d>=1.4.0','mamba-ssm']:
    r = subprocess.run([sys.executable,'-m','pip','install','-q',pkg,'--no-build-isolation'],
                       capture_output=True, text=True)
    print(f"{'OK' if r.returncode==0 else 'WARN'}: {pkg}")

print('DONE! -> Runtime -> Restart session -> then run Cells 2,3,4 only')


In [ ]:
# Cell 2 - Download all files (model code, weights, MediaPipe, UI static files)
import os, sys, zipfile, urllib.request, gdown

# Google Drive File IDs
ZIP_ID      = '1KNRow_erm0kYJUu0xrPlYxAwirhVCHkc'  # gm_grefel_demo_src.zip
WEIGHT_ID   = '1JZg_z9G6ClAEd-BtSLfVtv6zABoZZ4vT'  # best_model_fold_5.pth
STATIC_ID   = '1oLbrG_MOZofqDQkUdxEfjk5uCTLHD1Sx'  # live_demo_static.zip (fixed: window.location.origin API URLs)
MP_URL      = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'

WEIGHT_PATH = '/content/best_model_fold_5.pth'
MP_PATH     = '/tmp/face_landmarker.task'

# 1. Source code
if not os.path.exists('/content/gm_grefel_demo_src/model.py'):
    print('Downloading model source code...')
    gdown.download(id=ZIP_ID, output='/content/src.zip', quiet=False)
    with zipfile.ZipFile('/content/src.zip') as z: z.extractall('/content')
    print('  Source ready')
else:
    print('  Source already present')

# 2. Model weights
if not os.path.exists(WEIGHT_PATH):
    print('Downloading model weights (~1.2 GB)...')
    gdown.download(id=WEIGHT_ID, output=WEIGHT_PATH, quiet=False)
    print('  Weights ready')
else:
    print('  Weights already present')

# 3. UI static files
if not os.path.exists('/content/live_demo_static/index.html'):
    print('Downloading UI static files...')
    gdown.download(id=STATIC_ID, output='/content/static.zip', quiet=False)
    with zipfile.ZipFile('/content/static.zip') as z: z.extractall('/content')
    print('  Static UI ready')
else:
    print('  Static UI already present')

# 4. MediaPipe
if not os.path.exists(MP_PATH):
    print('Downloading MediaPipe...')
    urllib.request.urlretrieve(MP_URL, MP_PATH)
    print('  MediaPipe ready')
else:
    print('  MediaPipe already present')

# Patch index.html so API URLs work with any public ngrok domain
HTML_PATH = '/content/live_demo_static/index.html'
with open(HTML_PATH, 'r') as f: html = f.read()
html = html.replace("'http://localhost:5050/predict'", "window.location.origin+'/predict'")
html = html.replace("'http://localhost:5050/predict_image'", "window.location.origin+'/predict_image'")
with open(HTML_PATH, 'w') as f: f.write(html)

sys.path.insert(0, '/content/gm_grefel_demo_src')
print('All files ready - proceed to Cell 3')


In [ ]:
# Cell 3 - Write Flask backend to disk
FLASK_CODE = "import base64, io, sys, os\nimport cv2, numpy as np, torch, torch.nn.functional as F\nfrom flask import Flask, jsonify, request, send_from_directory\nfrom flask_cors import CORS\nfrom PIL import Image\nfrom torchvision import transforms\nimport mediapipe as mp\nfrom mediapipe.tasks import python as mp_python\nfrom mediapipe.tasks.python.vision import FaceLandmarker, FaceLandmarkerOptions, RunningMode\nsys.path.insert(0, '/content/gm_grefel_demo_src')\nfrom model import create_landmark_enhanced_efficientfer_ssm_dfew\n\nEMOTIONS  = ['Happy','Sad','Neutral','Angry','Surprise','Disgust','Fear']\nEMOJIS    = ['😄','😢','😐','😠','😲','🤢','😨']\nDEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nNUM_FRAMES= 16\n\nTRANSFORM = transforms.Compose([\n    transforms.Resize((224,224)), transforms.ToTensor(),\n    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),\n])\n\nprint(f'Loading GM-GReFEL on {DEVICE}...')\nmodel = create_landmark_enhanced_efficientfer_ssm_dfew(num_classes=7).to(DEVICE)\nckpt  = torch.load('/content/best_model_fold_5.pth', map_location=DEVICE, weights_only=False)\nmodel.load_state_dict(ckpt.get('model_state_dict', ckpt), strict=False)\nmodel.eval()\nfor p in model.parameters(): p.requires_grad = True\n\n_cache = []\ntry:\n    def _h(m,i,o):\n        _cache.clear()\n        if isinstance(o,torch.Tensor): _cache.append(o.detach().cpu().float())\n    model.temporal.register_forward_hook(_h)\nexcept: pass\n\n_mp = FaceLandmarkerOptions(\n    base_options=mp_python.BaseOptions(model_asset_path='/tmp/face_landmarker.task'),\n    running_mode=RunningMode.IMAGE, num_faces=1,\n    min_face_detection_confidence=0.3, min_face_presence_confidence=0.3,\n    min_tracking_confidence=0.3, output_face_blendshapes=False,\n    output_facial_transformation_matrixes=False,\n)\nLAND = FaceLandmarker.create_from_options(_mp)\n\n_LIPS=[61,185,40,39,37,0,267,269,270,409,291,146,91,181,84,17,314,405,321,375]\n_LEY=[33,7,163,144,145,153,154,155,133]; _REY=[362,382,381,380,374,373,390,249,263]\n_LBR=[70,63,105,66,107,55,65,52,53,46]; _RBR=[336,296,334,293,300,285,295,282,283,276]\n_NOS=[168,6,197,195,5,4,1,19,94]\n_OVL=[10,338,297,332,284,251,389,356,454,323,361,288,397,365,379,378,400,377,152,148,176,149,150,136,172,58,132,93,234,127,162,21,54,103,67,109,10]\n\ndef _dc(img,lm,idx,c,t=1):\n    h,w=img.shape[:2]; n=len(lm)\n    pts=[(int(lm[i].x*w),int(lm[i].y*h)) for i in idx if i<n]\n    for a,b in zip(pts,pts[1:]): cv2.line(img,a,b,c,t,cv2.LINE_AA)\ndef _dd(img,lm,idx,c,r=2):\n    h,w=img.shape[:2]; n=len(lm)\n    for i in idx:\n        if i<n: cv2.circle(img,(int(lm[i].x*w),int(lm[i].y*h)),r,c,-1)\n\ndef draw_lm(rgb):\n    out=rgb.copy()\n    res=LAND.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb))\n    if not res.face_landmarks: return out\n    lm=res.face_landmarks[0]\n    for idx,c in [(_OVL,(200,200,200)),(_LBR,(0,220,0)),(_RBR,(0,220,0)),\n                  (_LEY,(0,220,255)),(_REY,(0,220,255)),(_NOS,(255,200,0)),(_LIPS,(0,120,255))]:\n        _dc(out,lm,idx,c)\n    for idx,c in [(_OVL,(180,180,180)),(_LEY,(0,220,255)),(_REY,(0,220,255)),\n                  (_LBR,(0,255,0)),(_RBR,(0,255,0)),(_NOS,(255,200,0)),(_LIPS,(0,120,255))]:\n        _dd(out,lm,idx,c)\n    return out\n\ndef gcam(ft,pred):\n    ft2=ft.clone().detach().requires_grad_(True).to(DEVICE)\n    model.zero_grad(); model(ft2)[0,pred].backward()\n    g=ft2.grad.detach().cpu().numpy()[0]\n    masks=np.mean(np.abs(g),axis=1)\n    for i in range(len(masks)):\n        if masks[i].max()>0:\n            masks[i]=cv2.GaussianBlur(masks[i],(31,31),0)\n            masks[i]=(masks[i]-masks[i].min())/(masks[i].max()-masks[i].min()+1e-8)\n    return masks\n\ndef overlay(rgb,mask):\n    h,w=rgb.shape[:2]; m=cv2.resize(mask,(w,h))\n    hm=cv2.applyColorMap(np.uint8(255*m),cv2.COLORMAP_JET)\n    hm=cv2.cvtColor(hm,cv2.COLOR_BGR2RGB).astype(np.float32)/255.\n    return np.uint8(np.clip(0.55*hm+0.45*rgb.astype(np.float32)/255.,0,1)*255)\n\ndef b64(arr):\n    buf=io.BytesIO(); Image.fromarray(arr).save(buf,format='PNG')\n    return 'data:image/png;base64,'+base64.b64encode(buf.getvalue()).decode()\n\ndef dec(s):\n    return np.array(Image.open(io.BytesIO(base64.b64decode(s.split(',')[-1]))).convert('RGB'))\n\ndef ssm():\n    if not _cache: return [0.5]*16\n    out=_cache[0]; out=out[0] if out.dim()==3 else out\n    v=out.norm(dim=-1).numpy().astype(float)\n    v=(v-v.min())/(v.max()-v.min()+1e-8)\n    return np.interp(np.linspace(0,1,16),np.linspace(0,1,len(v)),v).tolist()\n\ndef infer(raw):\n    while len(raw)<NUM_FRAMES: raw=(raw*2)[:NUM_FRAMES]\n    raw=raw[:NUM_FRAMES]\n    ft=torch.stack([TRANSFORM(Image.fromarray(f)) for f in raw]).unsqueeze(0).to(DEVICE)\n    with torch.no_grad(): probs=F.softmax(model(ft),dim=1)[0].cpu().numpy()\n    pred=int(probs.argmax()); mid=cv2.resize(raw[NUM_FRAMES//2],(224,224))\n    masks=gcam(ft,pred)\n    return dict(emotion=EMOTIONS[pred],emoji=EMOJIS[pred],\n        probabilities={EMOTIONS[i]:float(f'{probs[i]*100:.2f}') for i in range(7)},\n        heatmap=b64(overlay(mid,masks[NUM_FRAMES//2])),\n        landmark=b64(draw_lm(mid)), ssm_curve=ssm())\n\napp=Flask(__name__, static_folder='/content/live_demo_static', static_url_path='')\nCORS(app)\n\n@app.route('/')\ndef index(): return send_from_directory('/content/live_demo_static','index.html')\n\n@app.route('/<path:f>')\ndef sf(f): return send_from_directory('/content/live_demo_static',f)\n\n@app.route('/predict', methods=['POST'])\ndef predict():\n    try:\n        p=request.get_json(force=True)\n        return jsonify(infer([dec(b) for b in p.get('frames',[])]))\n    except Exception as e:\n        import traceback; traceback.print_exc(); return jsonify(error=str(e)),500\n\n@app.route('/predict_image', methods=['POST'])\ndef predict_image():\n    try:\n        p=request.get_json(force=True)\n        raw=np.array(Image.open(io.BytesIO(base64.b64decode(p.get('image','').split(',')[-1]))).convert('RGB'))\n        return jsonify(infer([raw]*NUM_FRAMES))\n    except Exception as e:\n        import traceback; traceback.print_exc(); return jsonify(error=str(e)),500\n\nif __name__=='__main__':\n    app.run(host='0.0.0.0', port=5050, debug=False, threaded=False)\n"
with open('/content/colab_app.py', 'w') as f:
    f.write(FLASK_CODE)
print('Flask backend written to /content/colab_app.py')


In [ ]:
# Cell 4 - Start Flask server + expose via ngrok
import os, threading, time, sys, urllib.request
from pyngrok import ngrok

def run_flask():
    os.system(f'{sys.executable} /content/colab_app.py 2>&1')

t = threading.Thread(target=run_flask, daemon=True)
t.start()
print('Flask starting and loading model (~30s)...')
time.sleep(35)

# Verify Flask is up
try:
    urllib.request.urlopen('http://localhost:5050/')
    print('Flask is running!')
except:
    print('Still loading, waiting more...')
    time.sleep(15)

pub = ngrok.connect(5050)
url = pub.public_url if hasattr(pub,'public_url') else str(pub)
print(f'DEMO IS LIVE!')
print(f'Public URL: {url}')
print('Login: admin / 1234')
print('The UI is 100% identical to localhost:5050')
